## This pipeline takes SET report output json files as input and outputs fine-tuning compatibe instruction-response csv file



Input JSON key list input:

```
set_name
timestamp
execution_time_seconds

configuration.connector_config
configuration.set_config
configuration.target_model
configuration.evaluation_model
configuration.elm_evaluation_used

summary.total_sets
summary.total_passed
summary.total_failed
summary.total_error
summary.total_pass_rate
summary.total_fail_rate
summary.ci_lower_bound
summary.ci_upper_bound

results[].vulnerability_subcategory
results[].total_runs
results[].passed
results[].failed
results[].error
results[].pass_rate
results[].fail_rate
results[].recommended_remediation
results[].SETs[]

results[].SETs[].set_id
results[].SETs[].prompt
results[].SETs[].response
results[].SETs[].status
results[].SETs[].reason
results[].SETs[].elm_evaluation

results[].SETs[].detections.vulnerability.detected
results[].SETs[].detections.vulnerability.matches
results[].SETs[].detections.refusal.detected
results[].SETs[].detections.refusal.matches
results[].SETs[].detections.partial_compliance.detected
results[].SETs[].detections.partial_compliance.matches
results[].SETs[].detections.suspicious_output.detected
results[].SETs[].detections.suspicious_output.matches

results[].SETs[].metadata.vulnerability_subcategory
results[].SETs[].metadata.attack_type
results[].SETs[].metadata.expected_behavior

```

In [5]:
import json
import requests
from pathlib import Path
import os
import random
import uuid
from datetime import datetime, timedelta
import csv

In [ ]:
csv_path = Path("C:/Users/nikke/GitHub/ai-pentest-report-finetuning-pipeline/data/instruction_output.csv")

## Summarize SET report to include only necessary information

If you want to summarize reports, place them into a folder "raw_runs"

The folder "summaries" will contain your randomized and shortened report.json files

In [ ]:
# Example descriptions
SET_descriptions = {
    "blank": "Sends an empty prompt.",
    "atkgen": "Automated red-team attack generation.",
    "av_spam_scanning": "Triggers malicious content signature outputs.",
    "continuation": "Tests if model continues undesirable words.",
    "dan": "DAN-style jailbreak and roleplay attacks.",
    "donotanswer": "Prompts models that should be refused.",
    "encoding": "Prompt injection via encoded text.",
    "gcg": "Appends adversarial suffix to disrupt system prompt.",
    "glitch": "SETs for glitch tokens causing odd behavior.",
    "grandma": "Social-engineering style persona prompt (grandma).",
    "goodside": "Riley Goodside-style adversarial prompts.",
    "leakreplay": "Checks if model replays training data.",
    "lmrc": "Language Model Risk Cards subset SETs.",
    "malwaregen": "Attempts to generate malware code.",
    "misleading": "Prompts that encourage false or misleading claims.",
    "packagehallucination": "Tries to elicit references to fake packages.",
    "promptinject": "Agency-style prompt-injection adversarial SETs.",
    "realtoxicityprompts": "Subset of RealToxicityPrompts tests.",
    "snowball": "Hallucination cascade SETs for complex questions.",
    "xss": "Checks for exfiltration or cross-site vulnerabilities."
}

base_data_dir = Path(os.pardir) / "data"

# Subfolders
raw_dir = base_data_dir / "raw_runs"
processed_dir = base_data_dir / "processed_runs"
summaries_dir = base_data_dir / "summaries"

# Ensure directories exist
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
summaries_dir.mkdir(parents=True, exist_ok=True)

def summarize_SET_report(filetype, content, output_dir=summaries_dir):
    """
    Summarize a JSONL report and save the original into a processed folder.

    Parameters:
        filetype (str): "url" or "file"
        content (str): URL or file path to the .jsonl report
        output_dir (Path): Folder to save summarized JSON
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load entries
    if filetype == "url":
        response = requests.get(content)
        response.raise_for_status()
        entries = [json.loads(line) for line in response.text.splitlines()]
    elif filetype == "file":
        file_path = Path(content)
        entries = [json.loads(line) for line in file_path.read_text().splitlines()]
    else:
        raise ValueError("filetype must be either 'url' or 'file'")

    # Extract setup and evaluation results
    setup = next((e for e in entries if e.get("entry_type") == "start_run setup"), {})
    init = [e for e in entries if e.get("entry_type") == "init"]
    completion = [e for e in entries if e.get("entry_type") == "completion"]
    evals = [e for e in entries if e.get("entry_type") == "eval"]

    # Calculate run length
    start = datetime.fromisoformat(init[0].get("start_time")) if init else None
    try:
        end = datetime.fromisoformat(completion[0].get("end_time"))
        run_length = end - start
        minutes = run_length.total_seconds() / 60
        runtime = f"{run_length} ({minutes:.0f} minutes)"
    except (IndexError, TypeError, AttributeError):
        runtime = f"Started at {start.isoformat()}" if start else "Unknown runtime"

    eval_results = {}
    for eval in evals[:15]:
        SET = eval.get("SET", "unknown")
        category = SET.split('.')[0]
        if SET not in eval_results:
            eval_results[SET] = {
                "SET": SET,
                "description": SET_descriptions.get(category, "No description available."),
                "detectors": []
            }
        passed = eval.get("passed", 0)
        total = eval.get("total", 0)
        percentage = (passed / total * 100) if total else 0.0

        eval_results[SET]["detectors"].append({
            "detector": eval.get("detector"),
            "passed_count": passed,
            "total_count": total,
            "pass_percentage": f"{percentage:.1f}%",
            "outcome": "Resisted" if percentage >= 90 else "Vulnerable"
        })

    summary = {
        "run_id": setup.get("transient.run_id"),
        "model_type": setup.get("plugins.model_type"),
        "model_name": setup.get("plugins.model_name"),
        "run_length": runtime,
        "SETs": [
            {
                "SET_classname": SET,
                "description": info["description"],
                "evaluation_results": info["detectors"]
            }
            for SET, info in eval_results.items()
        ]
    }

    # Save summarized report
    filename = f"{setup.get('transient.run_id', 'unknown')}.summary.json"
    output_path = output_dir / filename
    with open(output_path, "w") as f:
        json.dump(summary, f, indent=2)

    return summary

## Randomize report contents

If you have reports that are only using one AI model and you want to duplicate those entries, use this

In [ ]:

model_choices = [
    "mistral", "llama3", "phi3", "gemma2", "qwen2", "mixtral",
    "yi", "command-r", "deepseek", "orca-mini"
]

def randomize_report(file_path, output_dir=processed_dir):
    new_model = random.choice(model_choices)
    new_run_id = str(uuid.uuid4())

    file_path = Path(file_path)
    output_file = Path(output_dir) / file_path.name

    with open(file_path, "r", encoding="utf-8") as infile, open(output_file, "w") as outfile:
        for line in infile:
            entry = json.loads(line)

            # Randomize model name and run ID
            if "plugins.model_name" in entry:
                entry["plugins.model_name"] = new_model
            if "transient.run_id" in entry:
                entry["transient.run_id"] = new_run_id
            if "transient.report_filename" in entry:
                entry["transient.report_filename"] = f"/root/.local/share/garak/garak_runs/garak.{new_run_id}.report.jsonl"
            if "run" in entry and isinstance(entry["run"], str) and len(entry["run"]) > 20:
                entry["run"] = new_run_id

            # Randomize nested meta/setup if present
            if "meta" in entry:
                meta = entry["meta"]
                if "setup" in meta and isinstance(meta["setup"], dict):
                    setup = meta["setup"]
                    if "plugins.model_name" in setup:
                        setup["plugins.model_name"] = new_model
                    if "transient.run_id" in setup:
                        setup["transient.run_id"] = new_run_id
                    if "transient.report_filename" in setup:
                        setup["transient.report_filename"] = f"/root/.local/share/garak/garak_runs/garak.{new_run_id}.report.jsonl"

            json.dump(entry, outfile)
            outfile.write("\n")

## Increase summary dataset size

Fabricates shortened and randomized runs artificially.

Selects 1-10 SETs and 1-3 detectors for each entry. Output is directed to "generated_runs"

In [16]:
SET_POOL = [
    # ANSI / Escape Handling
    {"vulnerability_subcategory": "Instruction Override", "classname": "ansiescape.AnsiEscaped", "description": "Checks model handling of ANSI escape sequences.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "ansiescape.AnsiRaw", "description": "Evaluates raw ANSI handling and escape parsing.", "remediation": "Input validation and normalization"},

    # Toxicity / Harmful Content
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "atkgen.Tox", "description": "Tests toxic or inflammatory prompt generation patterns.", "remediation": "Intent detection and output filtering"},

    # Audio / Multimodal Edge Cases
    {"vulnerability_subcategory": "Direct Injection", "classname": "audio.AudioAchillesHeel", "description": "Attempts to exploit audio-format ambiguities.", "remediation": "Input validation and format enforcement"},

    # AV / Spam / Malware Signatures
    {"vulnerability_subcategory": "Direct Injection", "classname": "av_spam_scanning.EICAR", "description": "Classic anti-virus test string SET.", "remediation": "Pattern-based input filtering"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "av_spam_scanning.GTUBE", "description": "Spam-filter evasion test string.", "remediation": "Pattern-based input filtering"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "av_spam_scanning.GTphish", "description": "Phishing-oriented spam signature test.", "remediation": "Intent detection and input filtering"},

    # Continuation / Slur Boundary Tests
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "continuation.ContinueSlursReclaimedSlurs", "description": "Continuation SET involving reclaimed slurs.", "remediation": "Continuation boundary enforcement"},
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "continuation.ContinueSlursReclaimedSlursFull", "description": "Extended continuation SET with reclaimed slurs.", "remediation": "Continuation boundary enforcement"},

    # DAN / Jailbreak SETs
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.Ablation_Dan_11_0", "description": "DAN jailbreak stress test.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.AutoDAN", "description": "Automated DAN jailbreak pattern.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.AutoDANCached", "description": "Cached AutoDAN jailbreak attempt.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.DanInTheWild", "description": "Observed live DAN jailbreak structure.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.DanInTheWildFull", "description": "Full variant of observed DAN jailbreak.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.ChatGPT_Image_Markdown", "description": "Jailbreak patterns involving image and Markdown cues.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.DAN_Jailbreak", "description": "Generalized DAN jailbreak attempt.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.DUDE", "description": "DUDE persona jailbreak.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.Dan_6_0", "description": "Early DAN jailbreak variant.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.Dan_6_2", "description": "Revised DAN jailbreak variant.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.Dan_7_0", "description": "Later-stage DAN jailbreak variant.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.Dan_8_0", "description": "Expanded DAN jailbreak attempt.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.Dan_9_0", "description": "Evolving DAN jailbreak pattern.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.ChatGPT_Developer_Mode_RANTI", "description": "Developer Mode jailbreak with RANTI pattern.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.ChatGPT_Developer_Mode_v2", "description": "Second version of Developer Mode jailbreak.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "dan.STAN", "description": "STAN persona jailbreak leveraging rule inversion.", "remediation": "Instruction hierarchy enforcement"},

    # Divergence / Repetition
    {"vulnerability_subcategory": "Indirect Injection", "classname": "divergence.Repeat", "description": "Repetition divergence stress pattern.", "remediation": "Output consistency controls"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "divergence.RepeatExtended", "description": "Extended repetition divergence test.", "remediation": "Output consistency controls"},

    # Encoding / Obfuscation Injection
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectAscii85", "description": "Tests ASCII85 injection resilience.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectBase16", "description": "Tests Base16 injection and parsing issues.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectBase2048", "description": "Evaluates high-radix Base2048 injection.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectBase32", "description": "Checks Base32 injection handling.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectBase64", "description": "Checks Base64 injection robustness.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectBraille", "description": "Encodes payloads in Unicode Braille patterns.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectEcoji", "description": "Ecoji emoji-based encoding stress test.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectHex", "description": "Hexadecimal injection evaluation.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectMorse", "description": "Morse-encoded payload injection test.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectNato", "description": "NATO phonetic-encoded injection test.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectROT13", "description": "ROT13-transformed payload injection.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectUU", "description": "UUEncoded payload injection test.", "remediation": "Input validation and normalization"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "encoding.InjectZalgo", "description": "Zalgo text corruption injection.", "remediation": "Input validation and normalization"},

    # Exploitation / Injection Simulation
    {"vulnerability_subcategory": "Direct Injection", "classname": "exploitation.JinjaTemplatePythonInjection", "description": "Template injection evaluations.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "exploitation.SQLInjectionEcho", "description": "SQL echo-based injection SET.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Direct Injection", "classname": "exploitation.SQLInjectionSystem", "description": "System-level SQL injection simulation.", "remediation": "Prompt boundary enforcement"},

    # Glitch / Destabilization
    {"vulnerability_subcategory": "Indirect Injection", "classname": "glitch.Glitch", "description": "General glitch pattern adversarial SET.", "remediation": "Output consistency controls"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "glitch.GlitchFull", "description": "Full glitch sequence for destabilization.", "remediation": "Output consistency controls"},

    # Goodside / Tag-Based Adversarial Prompts
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "goodside.Davidjl", "description": "Davidjl-style tagging adversarial sample.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "goodside.Tag", "description": "Goodside-style tagging adversarial test.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "goodside.ThreatenJSON", "description": "JSON-structured threatening pattern SET.", "remediation": "Intent detection and input filtering"},
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "goodside.WhoIsRiley", "description": "Identity-confusion tagging pattern.", "remediation": "Intent detection and input filtering"},

    # Persona-Based Jailbreaks (Grandma Family)
    {"vulnerability_subcategory": "Instruction Override", "classname": "grandma.Slurs", "description": "Grandma persona SET involving slur handling.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "grandma.Substances", "description": "Grandma persona SET involving substances.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "grandma.Win10", "description": "Windows-themed anthropomorphic jailbreak SET.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Instruction Override", "classname": "grandma.Win11", "description": "Windows 11 variant of persona-based jailbreak.", "remediation": "Instruction hierarchy enforcement"},

    # Latent Injection
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionFactSnippetEiffel", "description": "Latent fact-snippet injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionFactSnippetEiffelFull", "description": "Full Eiffel fact-snippet latent injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionFactSnippetLegal", "description": "Latent legal fact-snippet injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionFactSnippetLegalFull", "description": "Full legal latent fact-snippet injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionReport", "description": "Latent report-format injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionReportFull", "description": "Full report-format latent injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionResume", "description": "Latent résumé-format injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionResumeFull", "description": "Full résumé latent injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionTranslationEnFr", "description": "Latent English–French translation injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionTranslationEnFrFull", "description": "Full English–French latent injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionTranslationEnZh", "description": "Latent English–Chinese translation injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentInjectionTranslationEnZhFull", "description": "Full English–Chinese latent injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentJailbreak", "description": "Latent jailbreak pattern.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentJailbreakFull", "description": "Full latent jailbreak sequence.", "remediation": "Instruction hierarchy enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentWhois", "description": "Latent WHOIS-style identity injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentWhoisSnippet", "description": "WHOIS snippet latent injection.", "remediation": "Prompt boundary enforcement"},
    {"vulnerability_subcategory": "Latent Injection", "classname": "latentinjection.LatentWhoisSnippetFull", "description": "Full WHOIS snippet latent injection.", "remediation": "Prompt boundary enforcement"},

    # Leak / Replay / Reconstruction
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.GuardianCloze", "description": "News-text reconstruction leak test.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.GuardianClozeFull", "description": "Full Guardian cloze leak test.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.GuardianComplete", "description": "Guardian text reconstruction SET.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.GuardianCompleteFull", "description": "Full Guardian reconstruction SET.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.LiteratureCloze", "description": "Literary cloze leak evaluation.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.LiteratureClozeFull", "description": "Extended literary cloze reconstruction.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.LiteratureComplete", "description": "Literary text reconstruction SET.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.LiteratureCompleteFull", "description": "Full literary reconstruction SET.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.NYTCloze", "description": "NYT cloze leak test.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.NYTClozeFull", "description": "Extended NYT cloze leak test.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.NYTComplete", "description": "NYT reconstruction test.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.NYTCompleteFull", "description": "Full NYT reconstruction SET.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.PotterCloze", "description": "Harry Potter cloze leak test.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.PotterClozeFull", "description": "Extended Potter cloze leak test.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.PotterComplete", "description": "Potter reconstruction SET.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Indirect Injection", "classname": "leakreplay.PotterCompleteFull", "description": "Full Potter reconstruction SET.", "remediation": "Output content filtering"},

    # LMRC / Safety Policy Stress
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "lmrc.Bullying", "description": "Bullying language misuse SET.", "remediation": "Intent detection and output filtering"},
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "lmrc.Deadnaming", "description": "Ethical linguistic misuse SET.", "remediation": "Intent detection and output filtering"},
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "lmrc.Profanity", "description": "Profanity generation stress test.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "lmrc.QuackMedicine", "description": "Harmful medical-misinformation SET.", "remediation": "Intent detection and output filtering"},
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "lmrc.SexualContent", "description": "Sexual content boundary SET.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "lmrc.Sexualisation", "description": "Sexualisation pattern injection.", "remediation": "Output content filtering"},
    {"vulnerability_subcategory": "Roleplay Abuse", "classname": "lmrc.SlurUsage", "description": "Slur usage stress pattern.", "remediation": "Output content filtering"},
]


In [ ]:
taxonomy = {
  "language_model": {
    "input": {
      "single_turn": {

        "direct_prompt_injection": [

          // Persona / DAN / Instruction Override
          "Ablation_Dan_11_0",
          "AutoDAN",
          "AutoDANCached",
          "DanInTheWild",
          "DanInTheWildFull",
          "DAN_Jailbreak",
          "DUDE",
          "Dan_6_0",
          "Dan_6_2",
          "Dan_7_0",
          "Dan_8_0",
          "Dan_9_0",
          "ChatGPT_Developer_Mode_RANTI",
          "ChatGPT_Developer_Mode_v2",
          "STAN",
          "Slurs (Grandma)",
          "Substances (Grandma)",
          "Win10 (Grandma)",
          "Win11 (Grandma)",

          // Encoding / Obfuscation
          "InjectAscii85",
          "InjectBase16",
          "InjectBase2048",
          "InjectBase32",
          "InjectBase64",
          "InjectBraille",
          "InjectEcoji",
          "InjectHex",
          "InjectMorse",
          "InjectNato",
          "InjectROT13",
          "InjectUU",
          "InjectZalgo",

          // Format / Escape
          "AnsiEscaped",
          "AnsiRaw",
          "AudioAchillesHeel",

          // Template / SQL style
          "JinjaTemplatePythonInjection",
          "SQLInjectionEcho",
          "SQLInjectionSystem",

          // Signature payloads
          "EICAR",
          "GTUBE",
          "GTphish",

          // Explicit toxicity prompts
          "Tox",
          "Bullying",
          "Deadnaming",
          "Profanity",
          "QuackMedicine",
          "SexualContent",
          "Sexualisation",
          "SlurUsage",

          // Continuation boundary forcing
          "ContinueSlursReclaimedSlurs",
          "ContinueSlursReclaimedSlursFull",

          // Goodside-style adversarial prompts
          "Davidjl",
          "Tag",
          "ThreatenJSON",
          "WhoIsRiley"
        ],

        "indirect_prompt_injection": [

          // Latent contextual embedding
          "LatentInjectionFactSnippetEiffel",
          "LatentInjectionFactSnippetEiffelFull",
          "LatentInjectionFactSnippetLegal",
          "LatentInjectionFactSnippetLegalFull",
          "LatentInjectionReport",
          "LatentInjectionReportFull",
          "LatentInjectionResume",
          "LatentInjectionResumeFull",
          "LatentInjectionTranslationEnFr",
          "LatentInjectionTranslationEnFrFull",
          "LatentInjectionTranslationEnZh",
          "LatentInjectionTranslationEnZhFull",
          "LatentJailbreak",
          "LatentJailbreakFull",
          "LatentWhois",
          "LatentWhoisSnippet",
          "LatentWhoisSnippetFull",

          // Memorization / reconstruction leakage
          "GuardianCloze",
          "GuardianClozeFull",
          "GuardianComplete",
          "GuardianCompleteFull",
          "LiteratureCloze",
          "LiteratureClozeFull",
          "LiteratureComplete",
          "LiteratureCompleteFull",
          "NYTCloze",
          "NYTClozeFull",
          "NYTComplete",
          "NYTCompleteFull",
          "PotterCloze",
          "PotterClozeFull",
          "PotterComplete",
          "PotterCompleteFull",

          // Destabilization patterns
          "Repeat",
          "RepeatExtended",
          "Glitch",
          "GlitchFull"
        ]

      }
    }
  }
}


In [37]:
SUBCATEGORY_DESCRIPTIONS = [
    {"vulnerability_subcategory": "Instruction Override",
     "description": "Tests attempts to override, bypass, or manipulate model instructions, including jailbreak and persona-based exploits."},

    {"vulnerability_subcategory": "Roleplay Abuse",
     "description": "Evaluates misuse of model roleplay, tagging, or adversarial prompts to generate toxic, harmful, or boundary-violating content."},

    {"vulnerability_subcategory": "Direct Injection",
     "description": "Assesses susceptibility to direct payload injection, template execution, encoding, or malware/AV test strings."},

    {"vulnerability_subcategory": "Indirect Injection",
     "description": "Checks for indirect injection, output manipulation, repetition/divergence patterns, or content reconstruction attacks."},

    {"vulnerability_subcategory": "Latent Injection",
     "description": "Evaluates latent injection vectors, including hidden fact snippets, latent jailbreaks, and identity/translation manipulations."}
]


In [18]:
MODEL_POOL = [
    # --- Meta / Llama family ---
    "llama2",
    "llama3",
    "llama3.1",
    "llama3.2",
    "llama3.2-vision",
    "llama3.3",
    "llama4",

    # --- Mistral family ---
    "mistral",
    "mistral-nemo",
    "mistral-small",
    "mistral-small3.1",
    "mistral-small3.2",
    "mixtral",
    "codestral",

    # --- Qwen family ---
    "qwen",
    "qwen2",
    "qwen2.5",
    "qwen2.5-coder",
    "qwen2.5vl",
    "qwen3",
    "qwen3-coder",
    "qwen3-vl",
    "qwq",

    # --- Gemma family ---
    "gemma",
    "gemma2",
    "gemma3",
    "gemma3n",
    "codegemma",

    # --- DeepSeek family ---
    "deepseek-r1",
    "deepseek-v3",
    "deepseek-coder",
    "deepseek-coder-v2",

    # --- Phi family ---
    "phi",
    "phi3",
    "phi4",
    "phi4-mini",
    "phi4-reasoning",

    # --- IBM Granite ---
    "granite3.1-moe",
    "granite3.2-vision",
    "granite3.3",
    "granite4",
    "granite-code",

    # --- Dolphin variants ---
    "dolphin3",
    "dolphin-phi",
    "dolphin-llama3",
    "dolphin-mistral",
    "dolphin-mixtral",

    # --- Coding models ---
    "codellama",
    "starcoder2",
    "devstral",
    "deepcoder",

    # --- Vision / multimodal ---
    "llava",
    "llava-llama3",
    "minicpm-v",
    "moondream",

    # --- Other notable OSS models ---
    "falcon3",
    "olmo2",
    "orca-mini",
    "command-r",
    "wizardlm2",
    "hermes3",
    "openthinker",
    "magistral",
    "smollm",
    "smollm2",
    "tinyllama",
    "cogito",

    # --- OpenAI models ---
    "gpt-5",
    "gpt-5.2",
    "gpt-5.2-pro",
    "gpt-5-mini",
    "gpt-5-nano",
    "gpt-4.1",
    "gpt-oss-20b",
    "gpt-oss-120b"
]


In [ ]:
import os
import json
import uuid
import random
from pathlib import Path
import numpy as np

# --- Lookup tables ---
REMEDIATION_POOL = {entry["vulnerability_subcategory"]: entry["remediation"] for entry in SET_POOL}
SUBCATEGORY_DESC_LOOKUP = {entry["vulnerability_subcategory"]: entry["description"] for entry in SUBCATEGORY_DESCRIPTIONS}

# --- Output directory ---
base_data_dir = Path(os.pardir) / "data"
output_dir_generated = base_data_dir / "generated_runs"

# --- Functions ---

def sample_total_runs():
    bucket = random.choices(
        population=["small", "medium", "large", "very_large"],
        weights=[0.25, 0.35, 0.25, 0.15],
        k=1
    )[0]

    if bucket == "small":
        return random.randint(1, 10)
    elif bucket == "medium":
        return random.randint(10, 75)
    elif bucket == "large":
        return random.randint(75, 250)
    else:
        return random.randint(250, 600)


def generate_subcategory_entry(subcategory_name):
    total_runs = sample_total_runs()
    pass_rate = np.random.beta(2, 5)
    passed = int(pass_rate * total_runs)
    remaining = total_runs - passed
    error = random.randint(0, remaining)
    failed = remaining - error
    fail_rate = failed / total_runs if total_runs > 0 else 0
    status = (
        "secure" if fail_rate < 0.05 else
        "concerning" if fail_rate < 0.25 else
        "critical"
    )

    # Get all SETs of this subcategory
    sets_in_subcat_all = [s for s in SET_POOL if s["vulnerability_subcategory"] == subcategory_name]
    if not sets_in_subcat_all:
        # fallback if none found
        sets_in_subcat_all = [random.choice(SET_POOL)]

    # Always pick at least one SET randomly, then add random number of other SETs with same subcategory
    sets_count = random.randint(1, min(5, len(sets_in_subcat_all)))
    sets_in_subcat = random.sample(sets_in_subcat_all, sets_count)

    # Aggregate unique remediations only if status is concerning or critical
    remediations = list({s["remediation"] for s in sets_in_subcat}) if status != "secure" else ["No additional actions needed"]

    return {
        "subcategory": subcategory_name,
        "description": SUBCATEGORY_DESC_LOOKUP.get(subcategory_name, ""),
        "metrics": {
            "total_runs": total_runs,
            "passed": passed,
            "failed": failed,
            "error": error,
            "pass_rate": round(passed / total_runs, 4) if total_runs > 0 else 0,
            "fail_rate": round(fail_rate, 4)
        },
        "risk_assessment": {
            "status": status,
            "confidence_level": (
                "low" if total_runs < 10 else
                "medium" if total_runs < 100 else
                "high"
            )
        },
        "sub_category_remediations": remediations
    }



def generate_run():
    run_id = str(uuid.uuid4())
    run_length_seconds = random.randint(40, 600)

    # Random number of subcategories to pick
    subcategory_count = random.choices(
        population=[1, 2, 3, 4, 5],
        weights=[0.25, 0.30, 0.25, 0.15, 0.05],
        k=1
    )[0]

    selected_subcategories = []
    available_subcategories = VULNERABILITY_SUBCATEGORIES.copy()
    for _ in range(subcategory_count):
        if not available_subcategories:
            break
        subcat = random.choice(available_subcategories)
        selected_subcategories.append(subcat)
        # remove to avoid picking same subcategory twice in this run
        available_subcategories.remove(subcat)

    subcategories_data = [generate_subcategory_entry(name) for name in selected_subcategories]

    total_runs_all = sum(s["metrics"]["total_runs"] for s in subcategories_data)
    total_failed_all = sum(s["metrics"]["failed"] for s in subcategories_data)
    total_error_all = sum(s["metrics"]["error"] for s in subcategories_data)
    overall_fail_rate = total_failed_all / total_runs_all if total_runs_all > 0 else 0

    return {
        "run_metadata": {
            "run_id": run_id,
            "model_name": random.choice(MODEL_POOL),
            "execution_time_seconds": run_length_seconds
        },
        "aggregate_summary": {
            "total_subcategories": len(subcategories_data),
            "total_runs": total_runs_all,
            "total_failed": total_failed_all,
            "total_error": total_error_all,
            "overall_fail_rate": round(overall_fail_rate, 4),
            "overall_status": (
                "secure" if overall_fail_rate < 0.05 else
                "concerning" if overall_fail_rate < 0.25 else
                "critical"
            )
        },
        "subcategories": subcategories_data
    }


def increase_dataset_size(directory=output_dir_generated, amount=20):
    Path(directory).mkdir(parents=True, exist_ok=True)
    for _ in range(amount):
        run = generate_run()
        out_path = Path(directory) / f"{run['run_metadata']['run_id']}.generated.json"
        with open(out_path, "w", encoding="utf8") as f:
            json.dump(run, f, indent=2)
    print(f"Generated {amount} synthetic evaluation summaries in {directory}")


## Generate csv file from summarized and generated outputs

Selects data from "summaries" and "generated runs" folders and outputs csv into "data" with name "instruction_output.csv"

In [24]:
def generate_csv(input_data):
    """
    Generate a CSV with two columns:
    - instruction: the raw JSON or JSONL content
    - output: a human-readable summary of the report
    """
    folder_a, folder_b = input_data
    files_to_process = []

    for folder in [folder_a, folder_b]:
        if os.path.isdir(folder):
            for name in os.listdir(folder):
                path = os.path.join(folder, name)
                if name.lower().endswith((".json", ".jsonl")):
                    files_to_process.append(path)

    if not files_to_process:
        print("No JSON or JSONL files found.")
        return None

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["instruction", "output"])
        writer.writeheader()

        for file_path in files_to_process:
            with open(file_path, "r", encoding="utf-8") as fr:
                content = fr.read().strip()

            # Determine JSON or JSONL
            if file_path.lower().endswith(".json"):
                try:
                    reports = [json.loads(content)]
                except Exception as e:
                    print(f"Skipping invalid JSON {file_path}: {e}")
                    continue
            else:  # JSONL
                reports = []
                for line in content.splitlines():
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        reports.append(json.loads(line))
                    except Exception as e:
                        print(f"Skipping invalid JSONL line in {file_path}: {e}")
                        continue

            for report in reports:
                output_text = generate_report_string(report)
                writer.writerow({
                    "instruction": content,
                    "output": output_text
                })

    return csv_path


## Run code

In [ ]:
# Randomize all JSONL files in the raw folder
#for file in os.listdir(raw_dir):
#    file_path = os.path.join(raw_dir, file)
#    if os.path.isfile(file_path) and file.endswith(".jsonl"):
#        randomize_report(file_path)

In [ ]:
# Summarize all processed files
#for file in os.listdir(processed_dir):
#    file_path = os.path.join(processed_dir, file)
#    if os.path.isfile(file_path) and file.endswith(".jsonl"):
#        summarize_SET_report("file", file_path)

In [44]:
# Increase dataset size by amount entries
increase_dataset_size(amount=50)

Generated 50 synthetic evaluation summaries in ../data/generated_runs


In [ ]:
# Generate CSV from the two folders
# generate_csv((csv_path, output_dir_generated))